# 02. Kiểm thử Train/Test Split, Xử lý Âm & Tích hợp Đa Mô hình

Notebook này trình bày các đầu ra khi dùng `rfm-fe` cho các bài toán ML thực tế:
1. So sánh 4 chiến lược `handle_negative_amount`: `keep`, `clip_zero`, `drop`, `separate`.
2. Chia tập **Customer Split** và **Out-of-Time Split** không rò rỉ dữ liệu.
3. Đưa thẳng đầu ra vào **RandomForest** (Tree-based), **LogisticRegression** (Linear), và **KMeans** (Clustering).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from rfm_fe import RFMFeatureEngineer, engineer_rfm_features

In [2]:
def generate_rfm_test_data(n_rows=100_000, n_customers=10_000, seed=42):
    rng = np.random.default_rng(seed)
    weights = 1.0 / np.arange(1, n_customers + 1) ** 0.8
    weights /= weights.sum()
    customer_ids = rng.choice([f"CUST_{i:05d}" for i in range(1, n_customers + 1)], size=n_rows, p=weights)
    invoice_ids = [f"INV_{i:07d}" for i in range(1, n_rows + 1)]
    end_date = np.datetime64("2026-11-27")
    random_days = rng.integers(0, 365, size=n_rows)
    random_seconds = rng.integers(0, 86400, size=n_rows)
    invoice_dates = end_date - random_days.astype("timedelta64[D]") - random_seconds.astype("timedelta64[s]")
    quantities = rng.integers(1, 15, size=n_rows)
    unit_prices = np.round(rng.lognormal(mean=11.5, sigma=0.9, size=n_rows), -3)
    total_amounts = quantities * unit_prices
    statuses = rng.choice(["Completed", "Cancelled", "Refunded"], size=n_rows, p=[0.97, 0.02, 0.01])
    total_amounts = np.where(statuses == "Refunded", -total_amounts, total_amounts)
    df = pd.DataFrame({
        "invoice_id": invoice_ids,
        "customer_id": customer_ids,
        "invoice_date": invoice_dates,
        "quantity": quantities,
        "unit_price": unit_prices,
        "total_amount": total_amounts,
        "status": statuses,
    })
    return df.sort_values("invoice_date").reset_index(drop=True)

df = generate_rfm_test_data()
df_clean = df[df["status"] != "Cancelled"].copy()
print("df_clean shape:", df_clean.shape)

df_clean shape: (98001, 7)


## 1. So sánh 4 chiến lược xử lý giao dịch âm (`Refunded`)

In [3]:
rows = []
for strategy in ["keep", "clip_zero", "drop", "separate"]:
    out = engineer_rfm_features(
        df_clean,
        customer_col="customer_id",
        date_col="invoice_date",
        amount_col="total_amount",
        handle_negative_amount=strategy,
    )
    rows.append({
        "strategy": strategy,
        "n_customers": len(out),
        "n_columns": out.shape[1],
        "min_monetary_sum": out["monetary_sum"].min(),
        "mean_monetary_sum": round(out["monetary_sum"].mean(), 2),
        "has_refund_cols": "refund_count" in out.columns,
    })
pd.DataFrame(rows)

,strategy,n_customers,n_columns,min_monetary_sum,mean_monetary_sum,has_refund_cols
0,keep,9693,45,-9463000.0,10959937.07,False
1,clip_zero,9693,45,0.0,11076767.25,False
2,drop,9677,45,6000.0,11095081.64,False
3,separate,9677,48,6000.0,11095081.64,True


## 2. Kiểm thử Train/Test Split (Customer Split & Out-of-Time Split)

In [4]:
# 2a. Customer split
custs = df_clean["customer_id"].unique()
train_custs = set(custs[:8000])
df_tr_c = df_clean[df_clean["customer_id"].isin(train_custs)]
df_te_c = df_clean[~df_clean["customer_id"].isin(train_custs)]

fe_cust = RFMFeatureEngineer(
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    reference_date="2026-11-28",
    log_transform=True,
    scaler="standard",
    encode_segment=True,
)
X_tr_c = fe_cust.fit_transform(df_tr_c)
X_te_c = fe_cust.transform(df_te_c)
print("Customer split:", X_tr_c.shape, X_te_c.shape, "| Columns match:", list(X_tr_c.columns) == list(X_te_c.columns))

# 2b. Out-of-time split
df_tr_t = df_clean[df_clean["invoice_date"] <= "2026-08-31"]
df_te_t = df_clean[df_clean["invoice_date"] > "2026-08-31"]

fe_oot = RFMFeatureEngineer(
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    reference_date="2026-09-01",
    log_transform=True,
    scaler="robust",
)
X_tr_t = fe_oot.fit_transform(df_tr_t)
X_te_t = fe_oot.transform(df_te_t, reference_date="2026-11-28")
print("Out-of-time split:", X_tr_t.shape, X_te_t.shape, "| Columns match:", list(X_tr_t.columns) == list(X_te_t.columns))

Customer split: (8000, 54) (1693, 54) | Columns match: True


Out-of-time split: (9350, 45) (6609, 45) | Columns match: True


## 3. Huấn luyện thử trên Tree-based, Linear và Clustering (K-Means)

In [5]:
# Tạo dữ liệu cho Tree-based (giữ nguyên thang đo gốc)
X_tree = engineer_rfm_features(
    df_clean,
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    handle_negative_amount="separate",
    return_customer_col=False,
)
# Bài toán giả lập: dự đoán khách hàng thuộc top 30% tần suất mua
y_target = (X_tree["frequency"] >= X_tree["frequency"].quantile(0.7)).astype(int)
X_tree_no_leak = X_tree.drop(columns=["frequency", "f_score", "rfm_score_sum", "rfm_score_weighted"])

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tree_no_leak, y_target)
top_imp = pd.Series(rf.feature_importances_, index=X_tree_no_leak.columns).sort_values(ascending=False).head(8)
print(f"RandomForest acc: {rf.score(X_tree_no_leak, y_target):.4f}")
print("Top 8 features quan trọng nhất (RandomForest):")
display(top_imp.to_frame("importance"))

# Tạo dữ liệu cho Linear & K-Means (log_transform + StandardScaler)
X_scaled = engineer_rfm_features(
    df_clean,
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    handle_negative_amount="separate",
    log_transform=True,
    scaler="standard",
    encode_segment=True,
    return_customer_col=False,
)
X_scaled_no_leak = X_scaled.drop(columns=["frequency", "f_score", "rfm_score_sum", "rfm_score_weighted"])
lr = LogisticRegression(max_iter=500).fit(X_scaled_no_leak, y_target)
print(f"LogisticRegression acc: {lr.score(X_scaled_no_leak, y_target):.4f}")

# Phân cụm K-Means trên toàn bộ feature đã scale và xem profile gốc từng cụm
km = KMeans(n_clusters=4, random_state=42, n_init="auto")
clusters = km.fit_predict(X_scaled)
profile = X_tree.assign(cluster=clusters).groupby("cluster")[["recency_days", "frequency", "monetary_sum", "avg_order_value", "refund_rate"]].agg(["count", "mean"])
print("\nHồ sơ 4 cụm khách hàng từ K-Means (trên thang đo gốc):")
display(profile)

RandomForest acc: 1.0000
Top 8 features quan trọng nhất (RandomForest):


,importance
freq_365d,0.347398
active_months_count,0.112962
frequency_per_month,0.109878
freq_180d,0.077609
monetary_sum_365d,0.073029
monetary_sum,0.067957
active_span_days,0.028888
freq_90d,0.024914


LogisticRegression acc: 0.9998



Hồ sơ 4 cụm khách hàng từ K-Means (trên thang đo gốc):


recency_days             frequency            monetary_sum  \
               count        mean     count       mean        count   
cluster                                                              
0               2079   13.555431      2079  31.628668         2079   
1                899  177.823482       899   1.003337          899   
2               3983   44.158468      3983   5.543811         3983   
3               2716  140.228579      2716   3.035346         2716   

                      avg_order_value               refund_rate            
                 mean           count          mean       count      mean  
cluster                                                                    
0        3.571092e+07            2079  1.200204e+06        2079  0.010329  
1        1.059451e+06             899  1.058175e+06         899  0.017798  
2        6.114914e+06            3983  1.160688e+06        3983  0.007625  
3        2.877745e+06            2716  9.746336e+05        2716  0.007189